# HW 5: The Loan Officer Bot

**Your name:** Kamya Kanara

**Points:** 25

**Due:** November 13th at 21:00 (9pm)

**Submit:** `hw5_models.ipynb`, run top to bottom with outputs showing. 

**Double check:**
- *Make sure that you restart your kernel and run everything to do this. We will deduct points if your output isn't showing in your submitted file!*
- *Make sure all written answers are present*
- *Make sure that all functions' comments have been improved, as noted in the "Rules" section*

This homework should be completed individually. Follow the course's AI-use policy while completing it.

## The problem

A bank wants a model that decides whether to approve a loan. It has records of 1,000 past applicants, each labeled **good** or **bad** credit risk, and it asks you to find the best model for the job.

A fair lender should not treat applicants worse just because of their age. So before the bank uses any model, you need to check two things: **does it work**, and **is it fair to young applicants**?

You will compare **six models** on a real dataset: four classic models from scikit-learn, and two **small neural nets** that you build in PyTorch (one on top of GloVe word embeddings, one on top of BERT). For each model you will measure how it treats applicants **under 25 (`< 25`)** compared with applicants **25 and over**, and then test whether simply hiding the age column fixes any gap.

## Data

The **German Credit** dataset: 1,000 real loan applicants from a German bank (1994), loaded from OpenML with scikit-learn. The first load downloads it, so you need internet once.

- `class` is the label: `good` or `bad` credit risk. We use **1 = good (approve)** and **0 = bad (reject)**.
- The features include the loan amount and duration, checking and savings status, employment, housing, job, and `age`.
- We drop `personal_status` and `foreign_worker` from the start, because they record sex, marital status, and nationality. They are not used anywhere in this homework.
- The dataset's creators note that **approving a bad loan is 5 times worse than rejecting a good one**. You will use this cost in Task 2.

**Neural nets read text, not tables.** To give GloVe and BERT something to read, we describe each applicant in a short text, for example *"checking status is <0. duration is 12. purpose is used car. age is 24. ..."*. The function that does this is already written for you.

## Some important course/other connections

- **Task 3** (a small PyTorch net on top of `gensim` word embeddings) is very similar to the lecture activity from 10/27.
- **Task 4** keeps BERT frozen and trains a small net on top of its output vector, so the only new piece is getting the vector out of BERT.

## Functions you will write

| Function | Task | What it does |
|---|---|---|
| `audit_table(X, y, group)` | 1 | Summarizes the data for each age group |
| `build_models()` | 2 | Returns a dictionary of the baseline and four classic models |
| `summarize_predictions(y_true, pred, group)` | 2 | Turns one model's predictions into accuracy, cost, and fairness gaps |
| `compare_models(models, X, y, group, cv=5)` | 2 | Returns one table with a row per classic model |
| `average_glove(text, vectors)` | 3 | Turns a text into one GloVe vector |
| `LoanNetwork` and `train_net(...)` | 3 | A small PyTorch net and its training loop |
| `bert_embeddings(texts)` | 4 | Turns texts into BERT `[CLS]` vectors |

## Rules

- Use **Python**, **scikit-learn**, **PyTorch**, `gensim`, and `transformers` (`pip install scikit-learn numpy pandas torch gensim transformers`). We do not expect you to implement the "classic" models yourself.
- Use `random_state=0` (and `torch.manual_seed(0)`) everywhere.
- There is no separate test set. Every model is judged on **out-of-fold predictions** (each applicant is predicted by a model that never saw them), so there is nothing to tune on.
- Written answers go in the answer cell under each task.
- __The function comments we gave you are brief. Add to these so they are comprehensive enough that you understand what each function is doing. Make sure to include information on what the parameters and return values are/should be.__

Set-up Questions
---
1. How does the term __out-of-fold__ relate to train/dev/test set evaluation and k-fold evaluation?
- In k-fold evaluation the data is split into k folds, and each fold takes a turn as the held-out set while the model is trained on the other k-1 folds. The predictions made on each held-out fold are called out-of-fold predictions, because the model that made them never saw those applicants during training. Putting all k folds together gives one honest prediction for every one of the 1,000 applicants, so it plays the role of a test set for the whole dataset instead of setting aside a separate test set (which would waste data when we only have 1,000 rows). In a normal train/dev/test split the dev set is used for tuning and the test set for the final score; here there is no tuning (hyperparameters are fixed), so no dev set is needed and the out-of-fold predictions are the final evaluation.

2. Re-formulate, in your own words, what you'll be doing in this homework.
- I am building and comparing models that decide whether a loan applicant is a good or bad credit risk, and checking whether they treat young applicants (under 25) fairly. I train four classic scikit-learn models plus a majority-class baseline, and two small PyTorch neural nets that read a text description of each applicant (one using averaged GloVe word vectors, one using frozen BERT vectors). Every model is scored on out-of-fold predictions using accuracy, F1 and a cost that makes approving a bad loan 5 times worse than rejecting a good one. I also compute gaps in approval rates between age groups (demographic parity, true positive rate, false positive rate), and finally test whether deleting `age`, or `age` plus related features, actually removes the gap.

## Environment and certificate setup

This notebook expects a dedicated Python environment with a modern Transformers 4.x installation.
Do not install or upgrade packages from inside a running notebook cell.

For Windows/Anaconda, create the environment once from Anaconda Prompt:

```bash
conda create -n hw5 python=3.11 -y
conda activate hw5
python -m pip install numpy pandas scikit-learn torch gensim transformers==4.41.2 certifi ipykernel
python -m ipykernel install --user --name hw5 --display-name "Python (hw5)"
```

Then select **Python (hw5)** as the notebook kernel in VS Code.

The next cell configures verified HTTPS certificates. It does not disable certificate verification.


In [1]:
# Configure verified HTTPS certificates without changing Python's import paths.
import os
import ssl

try:
    import certifi
    os.environ["SSL_CERT_FILE"] = certifi.where()
    os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
    print("HTTPS certificate verification: using certifi's CA bundle.")
except ImportError:
    print("certifi is not installed; using Python's default certificate configuration.")

print("Python SSL default verify paths:", ssl.get_default_verify_paths())


HTTPS certificate verification: using certifi's CA bundle.
Python SSL default verify paths: DefaultVerifyPaths(cafile='c:\\Users\\hp\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\certifi\\cacert.pem', capath=None, openssl_cafile_env='SSL_CERT_FILE', openssl_cafile='C:\\Program Files\\Common Files\\SSL/cert.pem', openssl_capath_env='SSL_CERT_DIR', openssl_capath='C:\\Program Files\\Common Files\\SSL/certs')


In [2]:
# Imports and environment checks. Packages must be installed in the selected kernel's environment.
import re
import warnings
import numpy as np
import pandas as pd
import gensim.downloader
from IPython.display import display   # makes display() work in every Jupyter/VS Code kernel
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import transformers
try:
    from packaging.version import Version
except ImportError:                      # fall back to a tiny parser if 'packaging' is missing
    def Version(v):
        return tuple(int(p) for p in re.findall(r"\d+", v)[:3])

# The assignment uses tokenizer(texts, ...), which requires Transformers 4.x.
if Version(transformers.__version__) < Version("4.30.0"):
    raise RuntimeError(
        f"Transformers 4.30+ is required, but this kernel has "
        f"{transformers.__version__} at {transformers.__file__}. "
        "Stop here and select the dedicated 'Python (hw5)' kernel after "
        "creating it using the instructions above."
    )

from transformers import BertModel, BertTokenizer
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", message=".*ill-defined.*")  # the baseline triggers this warning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

# ---- the loan data ----
df = fetch_openml("credit-g", version=1, as_frame=True, parser="auto").frame.reset_index(drop=True)
y = (df["class"] == "good").astype(int)                                   # 1 = good credit risk (approve)
group = pd.Series(np.where(df["age"] < 25, "under 25", "25 and over"))     # used only to audit the models
X = df.drop(columns=["class", "personal_status", "foreign_worker"]).reset_index(drop=True)

COST_APPROVE_BAD = 5   # approving a bad loan
COST_REJECT_GOOD = 1   # rejecting a good applicant


def make_preprocessor():
    """Build fold-fitted preprocessing for mixed numeric and categorical data.

    Returns
    -------
    sklearn.compose.ColumnTransformer
        Numeric columns are standardized and nonnumeric columns are one-hot
        encoded with unseen-category tolerance. Because this transformer is
        inside each model Pipeline, it is fitted only on a training fold.
    """
    return ColumnTransformer([
        ("num", StandardScaler(), make_column_selector(dtype_include=np.number)),
        ("cat", OneHotEncoder(handle_unknown="ignore"), make_column_selector(dtype_exclude=np.number)),
    ])


def applicant_to_text(row, skip=()):
    """Describe one applicant as a short text, e.g. "duration is 12. purpose is used car. age is 24."

    Parameters
    ----------
    row : pandas.Series
        One applicant (one row of X); the index holds column names and the
        values hold that applicant's feature values.
    skip : iterable of str, default=()
        Column names to leave out of the description (used in Task 5 to
        hide `age` and its proxies from the neural nets).

    Returns
    -------
    str
        One sentence per remaining column in the form "<column name> is
        <value>." joined by spaces. Underscores in column names become
        spaces, and numeric values are formatted compactly (e.g. 12.0 -> "12").
    """
    parts = []
    for col, val in row.items():
        if col in skip:
            continue
        if isinstance(val, (int, float, np.number)):
            val = f"{val:g}"
        parts.append(f"{col.replace('_', ' ')} is {val}.")
    return " ".join(parts)


# ---- pretrained models used by the neural nets (the first run downloads about 500 MB) ----
GLOVE = gensim.downloader.load("glove-wiki-gigaword-50")     # 50-dimensional word vectors, as in lecture

# things we'll need later for BERT
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
bert.eval()
for p in bert.parameters():       # BERT stays frozen: we only use it to turn text into numbers
    p.requires_grad = False
MAX_LENGTH = 192

# Make sure we successfully loaded the data
print(X.shape, "applicants x features")
print(group.value_counts())
print(applicant_to_text(X.iloc[0]))

(1000, 18) applicants x features
25 and over    851
under 25       149
Name: count, dtype: int64
checking status is <0. duration is 6. credit history is critical/other existing credit. purpose is radio/tv. credit amount is 1169. savings status is no known savings. employment is >=7. installment commitment is 4. other parties is none. residence since is 4. property magnitude is real estate. age is 67. other payment plans is none. housing is own. existing credits is 2. job is skilled. num dependents is 1. own telephone is yes.


## Task 1: Audit the data

Before training anything, look at the data you're working with. Write `audit_table(X, y, group)`. It returns a DataFrame with **one row per age group** and these columns:

- `n`: the number of applicants in the group
- `good_credit_rate`: the fraction of the group labeled good credit (`y == 1`)
- the **mean of every numeric feature** in `X` (for example `duration`, `credit_amount`, `age`)

*Hint:* Print the table, and also print `pd.crosstab(group, X["employment"], normalize="index")` and the same for `housing`.

In [3]:
def audit_table(X, y, group):
    """Summarize credit outcomes and numeric features separately by age group.

    Parameters
    ----------
    X : pandas.DataFrame
        Applicant feature table; each row is one applicant.
    y : pandas.Series or array-like
        Binary outcome aligned row-for-row with X (1 = good credit, 0 = bad).
    group : pandas.Series or array-like
        Age-group label for each applicant ("under 25" or "25 and over").

    Returns
    -------
    pandas.DataFrame
        One row per age group. Columns include group size (`n`), the share
        labeled good (`good_credit_rate`), and means of all numeric features.

    Notes
    -----
    The function aligns y and group to X's index before grouping, so the
    summaries remain attached to the correct applicant rows.
    """
    y = pd.Series(np.asarray(y), index=X.index, name="label")
    group = pd.Series(np.asarray(group), index=X.index, name="age_group")
    out = pd.DataFrame({"n": group.value_counts(),
                        "good_credit_rate": y.groupby(group).mean()})
    numeric_means = X.select_dtypes(include=np.number).groupby(group).mean()
    out = out.join(numeric_means)
    return out.sort_index()


In [4]:
audit = audit_table(X, y, group)
print("Audit table by age group:")
display(audit.T)
print("\nemployment by age group (row-normalized):")
display(pd.crosstab(group, X["employment"], normalize="index").round(3))
print("\nhousing by age group (row-normalized):")
display(pd.crosstab(group, X["housing"], normalize="index").round(3))


Audit table by age group:


age_group,25 and over,under 25
n,851.000000,149.000000
good_credit_rate,0.719154,0.590604
duration,20.975323,20.489933
credit_amount,3325.972973,2958.758389
installment_commitment,3.008226,2.771812
residence_since,2.834313,2.906040
age,37.814336,22.590604
existing_credits,1.443008,1.201342
num_dependents,1.175088,1.040268



employment by age group (row-normalized):


employment,1<=X<4,4<=X<7,<1,>=7,unemployed
row_0,,,,,
25 and over,0.329,0.172,0.149,0.287,0.063
under 25,0.396,0.188,0.302,0.060,0.054



housing by age group (row-normalized):


housing,for free,own,rent
row_0,,,
25 and over,0.121,0.750,0.129
under 25,0.034,0.503,0.463


**Write (2 to 3 sentences):** No, the share labeled good credit is not the same: it is about 0.72 for applicants 25 and over versus 0.59 for applicants under 25, a gap of roughly 13 percentage points. Two non-age features that clearly differ are `housing` (75% of the older group own their home versus 50% of the younger group, while 46% of the younger group rent versus 13% of the older group) and `employment` (29% of the older group have been employed 7 or more years versus 6% of the younger group, and 30% of the younger group have been employed under 1 year versus 15%). Because these features are correlated with age, a model can use them as indirect signals of age even if `age` itself is hidden.

## Task 2: Classic models

**(a)** Write `build_models()`. It returns a dictionary that maps a model name to a scikit-learn `Pipeline` with two steps, named `"prep"` (use `make_preprocessor()` from the setup cell) and `"clf"` (the classifier). Include these five:

| Name | Classifier |
|---|---|
| Majority baseline | `DummyClassifier(strategy="most_frequent")` |
| Logistic regression (SGD) | `SGDClassifier(loss="log_loss", max_iter=1000)` |
| Linear SVM | `LinearSVC()` |
| k-nearest neighbors | `KNeighborsClassifier(n_neighbors=15)` |
| Random forest | `RandomForestClassifier(n_estimators=100)` |

Pass `random_state=SEED` to every classifier that accepts it.

**(b)** Write `summarize_predictions(y_true, pred, group)`. Here `pred` is a model's decision for every applicant (1 = approve). Return a dictionary with these keys:

| Key | Meaning |
|---|---|
| `accuracy` | fraction correct |
| `precision`, `recall` | for the **good** class (`1`) |
| `macro_f1` | `f1_score(..., average="macro")` |
| `cost_per_applicant` | `(5 x approved-but-bad + 1 x rejected-but-good) / number of applicants` |
| `dp_gap` | **demographic parity gap**: (approval rate for "under 25") minus (approval rate for "25 and over"), over **all** applicants |
| `tpr_gap` | the same difference, but only over applicants who truly are **good** (`y_true == 1`) |
| `fpr_gap` | the same difference, but only over applicants who truly are **bad** (`y_true == 0`) |

The last two together are the **equalized-odds** gaps. A **negative gap** means young applicants are approved less often.

**(c)** Write `compare_models(models, X, y, group, cv=5)`. For every model, get out-of-fold predictions for all 1,000 applicants with `cross_val_predict(clone(model), X, y, cv=StratifiedKFold(cv, shuffle=True, random_state=SEED))`, pass them to `summarize_predictions`, and return one DataFrame with one row per model and one column per key, sorted by `macro_f1` from highest to lowest. Print the table.

Set-up Questions (Task 2)
---

1. What is the sklearn `Pipeline` class? Write down some notes to help *you* understand what this class's purpose is and how it works here.
- A `Pipeline` chains several steps into one object: zero or more transformers followed by a final estimator. Here every pipeline has two named steps, `"prep"` (the `ColumnTransformer` that standardizes numeric columns and one-hot encodes categorical columns) and `"clf"` (the classifier).
- Calling `fit(X, y)` runs `fit_transform` on `prep` using the training data, then fits `clf` on the transformed output. Calling `predict(X)` applies the already-fitted `prep` to new data and then calls `clf.predict`.
- The main benefit is avoiding data leakage: when the pipeline is passed to `cross_val_predict`, the scaler and encoder are re-fit on each training fold only, so nothing about the held-out fold leaks into preprocessing.
- It also lets me treat "preprocessing + model" as a single estimator, so `clone`, `cross_val_predict`, and my `compare_models` function work the same way for every model, and swapping classifiers only means changing the `"clf"` step.

In [5]:
def build_models():
    """Create the five required classic scikit-learn classification pipelines.

    Returns
    -------
    dict[str, sklearn.pipeline.Pipeline]
        A mapping from readable model names to pipelines. Each pipeline has
        a `prep` step (numeric standardization and categorical one-hot
        encoding) followed by a `clf` step (the classifier). Preprocessing is
        fitted inside each cross-validation training fold to prevent leakage.
    """
    return {
        "Majority baseline": Pipeline([
            ("prep", make_preprocessor()),
            ("clf", DummyClassifier(strategy="most_frequent",
                                  random_state=SEED)),
        ]),
        "Logistic regression (SGD)": Pipeline([
            ("prep", make_preprocessor()),
            ("clf", SGDClassifier(loss="log_loss", max_iter=1000,
                                  random_state=SEED)),
        ]),
        "Linear SVM": Pipeline([
            ("prep", make_preprocessor()),
            ("clf", LinearSVC(random_state=SEED)),
        ]),
        "k-nearest neighbors": Pipeline([
            ("prep", make_preprocessor()),
            ("clf", KNeighborsClassifier(n_neighbors=15)),
        ]),
        "Random forest": Pipeline([
            ("prep", make_preprocessor()),
            ("clf", RandomForestClassifier(n_estimators=100,
                                            random_state=SEED)),
        ]),
    }


def summarize_predictions(y_true, pred, group):
    """Calculate predictive performance, asymmetric cost, and age-group gaps.

    Parameters
    ----------
    y_true : array-like of int
        Ground-truth labels (1 = good credit, 0 = bad credit).
    pred : array-like of int
        Binary model decisions (1 = approve, 0 = reject), aligned with y_true.
    group : array-like of str
        Age-group label per applicant: "under 25" or "25 and over".

    Returns
    -------
    dict[str, float]
        Accuracy, precision and recall for class 1, macro-F1, cost per
        applicant, demographic-parity gap, true-positive-rate gap, and
        false-positive-rate gap. Every gap is rate(under 25) minus
        rate(25 and over); negative values indicate a lower approval rate
        for the younger group for the corresponding population/subgroup.

    Notes
    -----
    Approving a bad applicant costs 5 and rejecting a good applicant costs 1.
    Undefined subgroup rates are represented as NaN rather than silently
    treating an empty subgroup as having a zero rate.
    """
    y_true = np.asarray(y_true, dtype=int)
    pred = np.asarray(pred, dtype=int)
    group = np.asarray(group)
    if not (len(y_true) == len(pred) == len(group)):
        raise ValueError("y_true, pred, and group must have the same length")
    if len(y_true) == 0:
        raise ValueError("Cannot summarize an empty set of predictions")

    precision, recall, _, _ = precision_recall_fscore_support(
        y_true, pred, average=None, labels=[0, 1], zero_division=0)
    approved_bad = np.sum((pred == 1) & (y_true == 0))
    rejected_good = np.sum((pred == 0) & (y_true == 1))
    cost = (COST_APPROVE_BAD * approved_bad +
            COST_REJECT_GOOD * rejected_good) / len(y_true)

    young = group == "under 25"
    older = group == "25 and over"
    def approval_rate(mask):
        return float(pred[mask].mean()) if np.any(mask) else np.nan

    good = y_true == 1
    bad = y_true == 0
    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision[1]),
        "recall": float(recall[1]),
        "macro_f1": float(f1_score(y_true, pred, average="macro")),
        "cost_per_applicant": float(cost),
        "dp_gap": approval_rate(young) - approval_rate(older),
        "tpr_gap": approval_rate(young & good) - approval_rate(older & good),
        "fpr_gap": approval_rate(young & bad) - approval_rate(older & bad),
    }


def compare_models(models, X, y, group, cv=5):
    """Evaluate models with stratified out-of-fold predictions.

    Parameters
    ----------
    models : dict[str, estimator]
        Names and unfitted sklearn estimators/pipelines supporting fit/predict.
    X : pandas.DataFrame
        Feature matrix; preprocessing belongs inside each pipeline.
    y : array-like
        Binary target labels.
    group : array-like
        Age-group labels used only to calculate fairness gaps.
    cv : int, default=5
        Number of stratified folds. Each applicant is predicted by a model
        trained without that applicant.

    Returns
    -------
    pandas.DataFrame
        One row per model and one column per metric from
        `summarize_predictions`, sorted by macro-F1 in descending order.
    """
    y_arr = np.asarray(y, dtype=int)
    folds = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    rows = {}
    for name, model in models.items():
        pred = cross_val_predict(clone(model), X, y_arr, cv=folds, n_jobs=None)
        rows[name] = summarize_predictions(y_arr, pred, group)
    return pd.DataFrame.from_dict(rows, orient="index").sort_values(
        "macro_f1", ascending=False)


models = build_models()
print("Models:", list(models))


Models: ['Majority baseline', 'Logistic regression (SGD)', 'Linear SVM', 'k-nearest neighbors', 'Random forest']


In [6]:
results = compare_models(models, X, y, group, cv=5)
print("Classic model out-of-fold results:")
display(results.round(3))
print("Best macro-F1:", results["macro_f1"].idxmax(),
      f"({results['macro_f1'].max():.3f})")
print("Lowest cost:", results["cost_per_applicant"].idxmin(),
      f"({results['cost_per_applicant'].min():.3f})")
print("Majority-baseline cost:",
      f"{results.loc['Majority baseline', 'cost_per_applicant']:.3f}")


Classic model out-of-fold results:


,accuracy,precision,recall,macro_f1,cost_per_applicant,dp_gap,tpr_gap,fpr_gap
Linear SVM,0.754,0.794,0.876,0.683,0.882,-0.150,-0.170,-0.007
Random forest,0.763,0.782,0.917,0.675,0.953,-0.129,-0.087,-0.090
Logistic regression (SGD),0.708,0.764,0.844,0.623,1.024,-0.105,-0.108,-0.025
k-nearest neighbors,0.728,0.736,0.953,0.570,1.228,-0.087,-0.076,-0.053
Majority baseline,0.700,0.700,1.000,0.412,1.500,0.000,0.000,0.000


Best macro-F1: Linear SVM (0.683)
Lowest cost: Linear SVM (0.882)
Majority-baseline cost: 1.500


**Write (~3 sentences per classifier):** We haven't described *linear SVM*, *K-nearest neighbor*, or *random forest* classifiers in class. For each of these classifiers, *briefly* describe what kind of classifier this is and how it works. Plan to write about 3 sentences per classifier. You do not need to include math equations (we recommend you do not, in fact).
- *linear SVM*: A linear support vector machine is a linear classifier that separates the two classes with a straight line (a hyperplane in many dimensions). It is trained to keep a margin between the boundary and the closest training points of each class, while penalizing points that fall on the wrong side or inside the margin (which is needed because real data like this is not perfectly separable). Those closest points (the support vectors) determine the boundary, and a new applicant is classified by which side of the line they fall on.
- *K-nearest neighbor*: K-nearest neighbors is a simple instance-based classifier that does not learn a model during training; it just stores the training data. To classify a new applicant it finds the k most similar stored applicants (here k = 15) by distance in the preprocessed feature space and takes a majority vote of their labels. Because it relies on distances, feature scaling matters, and it can be hurt when there are many one-hot features.
- *random forest*: A random forest is an ensemble of many decision trees (here 100). Each tree is trained on a random resample of the data and considers only a random subset of features at each split, so the trees make different errors. The forest predicts by majority vote across trees, which is usually more accurate and less prone to overfitting than a single tree.


**Write (2 to 3 sentences):** Linear SVM has both the best macro-F1 (0.683) and the lowest cost (0.882 per applicant), and every trained model beats the majority baseline's cost of 1.500 (random forest 0.953, SGD logistic regression 1.024, k-nearest neighbors 1.228). Every gap is negative, so young applicants are approved less often in all three views: overall (`dp_gap` from -0.087 to -0.150), among truly good applicants (`tpr_gap` from -0.076 to -0.170, largest for Linear SVM), and among truly bad applicants (`fpr_gap` from -0.007 to -0.090).

## Task 3: A small neural net on GloVe

Now add a model that reads text. This is very similar to the lecture activity from 10/27: turn text into a dense vector with `gensim`, then train a small PyTorch net on it. The difference is that instead of a single word, the input is a whole description of an applicant.

**(a)** Write `average_glove(text, vectors)`: lowercase the text, split it into words with `re.findall(r"[a-z0-9]+", text.lower())` or `nltk.word_tokenize`, keep the words that are present in your gensim vectors`vectors`, and return the **average** of their vectors (a 50-dimensional numpy array). If no word is in `vectors`, return a vector of 50 zeros.

**(b)** Write the network. `LoanNetwork(input_size, hidden_units)` is a feedforward net with an input layer (`input_size` to `hidden_units`), a ReLU, and an output layer (`hidden_units` to 1). Its `forward` returns the **logits** (do not apply a sigmoid, because `BCEWithLogitsLoss` does that for you). Then write `train_net(X_train, y_train, epochs=30, lr=1e-3, hidden_units=32, batch_size=32)`:

1. turn `X_train` and `y_train` into float tensors (`y_train` with shape `(n, 1)`) and wrap them in a `DataLoader` with `shuffle=True`,
2. create a `LoanNetwork`, an `optim.Adam` optimizer, and `nn.BCEWithLogitsLoss()`,
3. for each epoch and each batch: `zero_grad`, forward pass, loss, `backward`, `step`, and
4. return the trained net.

(The lecture activity used SGD; Adam just trains faster on this small dataset.)

**(c)** The code below turns the net into a model you can compare with the others: `net_oof_predictions` runs the **same 5 folds** as `compare_models`, scales the features, trains a fresh net on each fold, and returns predictions for every applicant. Describe every applicant with `applicant_to_text`, average their GloVe vectors, get predictions, and add a row named `"Neural net on GloVe"` to your `results` table from Task 2 with `results.loc["Neural net on GloVe"] = pd.Series(summarize_predictions(y, pred, group))`. Print the table.

**(d)** As in the lecture activity, test **three sizes of hidden layer**: smaller than the input (`hidden_units=8`), the same as the input (`50`), and larger (`200`). Print the `macro_f1` for each.

In [7]:
def average_glove(text, vectors):
    """Convert a text description into the mean of its known GloVe vectors.

    Parameters
    ----------
    text : str
        Applicant description generated by `applicant_to_text`.
    vectors : gensim.models.KeyedVectors
        Pretrained word vectors; this assignment uses 50-dimensional GloVe.

    Returns
    -------
    numpy.ndarray, shape (vectors.vector_size,)
        Float32 mean vector for in-vocabulary tokens. If no token is found,
        returns a zero vector with the same dimensionality.

    Notes
    -----
    Tokenization lowercases the text and retains alphanumeric word pieces.
    Repeated in-vocabulary tokens contribute repeatedly to the average.
    """
    dim = vectors.vector_size
    tokens = re.findall(r"[a-z0-9]+", str(text).lower())
    found = [vectors[token] for token in tokens if token in vectors]
    if not found:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(found, axis=0, dtype=np.float32).astype(np.float32)


class LoanNetwork(nn.Module):
    """Feed-forward binary classifier: input -> hidden ReLU -> one logit."""
    def __init__(self, input_size: int, hidden_units: int = 32):
        """Initialize the two learned linear layers.

        Parameters
        ----------
        input_size : int
            Number of input features (50 for GloVe or 768 for BERT).
        hidden_units : int, default=32
            Number of neurons in the hidden layer.
        """
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_units)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_units, 1)

    def forward(self, x):
        """Compute one unnormalized logit per example.

        Parameters
        ----------
        x : torch.Tensor, shape (batch_size, input_size)
            Batch of float feature vectors.

        Returns
        -------
        torch.Tensor, shape (batch_size, 1)
            Raw logits. No sigmoid is applied because BCEWithLogitsLoss
            combines the sigmoid and binary cross-entropy operations.
        """
        return self.fc2(self.relu(self.fc1(x)))


def train_net(X_train, y_train, epochs=30, lr=1e-3,
              hidden_units=32, batch_size=32):
    """Train a LoanNetwork on one training fold using mini-batch Adam.

    Parameters
    ----------
    X_train : array-like, shape (n_samples, n_features)
        Training feature matrix, already scaled by the caller when needed.
    y_train : array-like, shape (n_samples,)
        Binary training labels (0 or 1).
    epochs : int, default=30
        Number of complete passes over the training data.
    lr : float, default=1e-3
        Adam optimizer learning rate.
    hidden_units : int, default=32
        Width of the hidden layer.
    batch_size : int, default=32
        Number of examples per mini-batch.

    Returns
    -------
    LoanNetwork
        Trained network in evaluation mode.
    """
    X_t = torch.as_tensor(np.asarray(X_train), dtype=torch.float32)
    y_t = torch.as_tensor(np.asarray(y_train), dtype=torch.float32).reshape(-1, 1)
    loader = DataLoader(TensorDataset(X_t, y_t),
                        batch_size=batch_size, shuffle=True)
    net = LoanNetwork(input_size=X_t.shape[1], hidden_units=hidden_units)
    optimizer = optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    net.train()
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(net(xb), yb)
            loss.backward()
            optimizer.step()
    net.eval()
    return net


In [8]:
# Sanity checks for the GloVe representation and neural-network output shape.
v = average_glove("duration is 12. purpose is used car. age is 24.", GLOVE)
print("Average GloVe vector shape:", v.shape,
      "| contains nonzero values:", bool(np.any(v)))
print("Unknown-token fallback shape:", average_glove("zzzz qqqq", GLOVE).shape)
net_check = LoanNetwork(input_size=50, hidden_units=32)
print("Network output shape:", tuple(net_check(torch.zeros(4, 50)).shape))


Average GloVe vector shape: (50,) | contains nonzero values: True
Unknown-token fallback shape: (50,)
Network output shape: (4, 1)


In [9]:
# ---- prediction and out-of-fold evaluation helpers for the neural net ----
def predict_net(net, X):
    """Convert a trained network's logits into binary class predictions.

    Parameters
    ----------
    net : LoanNetwork
        Trained binary classifier.
    X : array-like, shape (n_samples, n_features)
        Feature matrix already transformed with the scaler fitted on the
        corresponding training fold.

    Returns
    -------
    numpy.ndarray, shape (n_samples,)
        Integer predictions using a probability threshold of 0.5.
    """
    net.eval()
    with torch.no_grad():
        logits = net(torch.as_tensor(np.asarray(X), dtype=torch.float32))
    return (torch.sigmoid(logits).squeeze(1).cpu().numpy() >= 0.5).astype(int)


def net_oof_predictions(features, y, cv=5, **train_kwargs):
    """Generate one honest out-of-fold prediction for every applicant.

    Parameters
    ----------
    features : array-like, shape (n_samples, n_features)
        Numeric feature vectors for the applicants.
    y : array-like, shape (n_samples,)
        Binary credit labels.
    cv : int, default=5
        Number of stratified folds, using the same seed and split strategy
        as the classic-model comparison.
    **train_kwargs
        Optional keyword arguments forwarded to `train_net`, such as
        `epochs`, `lr`, `hidden_units`, and `batch_size`.

    Returns
    -------
    numpy.ndarray, shape (n_samples,)
        Binary predictions in the original applicant order. Each prediction
        comes from a network trained without that applicant.

    Notes
    -----
    A fresh StandardScaler and network are fitted for each fold. Scaling is
    learned only from the fold's training rows to prevent leakage.
    """
    features = np.asarray(features, dtype=np.float32)
    y = np.asarray(y, dtype=int)
    pred = np.zeros(len(y), dtype=int)
    splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    for train_idx, test_idx in splitter.split(features, y):
        scaler = StandardScaler().fit(features[train_idx])
        torch.manual_seed(SEED)
        net = train_net(scaler.transform(features[train_idx]),
                        y[train_idx], **train_kwargs)
        pred[test_idx] = predict_net(net, scaler.transform(features[test_idx]))
    return pred


In [10]:
# Describe each applicant, average its GloVe word vectors, and evaluate out of fold.
texts = [applicant_to_text(row) for _, row in X.iterrows()]
glove_feats = np.vstack([average_glove(text, GLOVE) for text in texts])
print("GloVe feature matrix:", glove_feats.shape)

pred_glove = net_oof_predictions(glove_feats, y, cv=5)
results.loc["Neural net on GloVe"] = pd.Series(
    summarize_predictions(y, pred_glove, group))
results = results.sort_values("macro_f1", ascending=False)
print("All classic models plus GloVe:")
display(results.round(3))


GloVe feature matrix: (1000, 50)
All classic models plus GloVe:


,accuracy,precision,recall,macro_f1,cost_per_applicant,dp_gap,tpr_gap,fpr_gap
Linear SVM,0.754,0.794,0.876,0.683,0.882,-0.150,-0.170,-0.007
Random forest,0.763,0.782,0.917,0.675,0.953,-0.129,-0.087,-0.090
Neural net on GloVe,0.725,0.782,0.841,0.654,0.931,-0.080,-0.092,0.034
Logistic regression (SGD),0.708,0.764,0.844,0.623,1.024,-0.105,-0.108,-0.025
k-nearest neighbors,0.728,0.736,0.953,0.570,1.228,-0.087,-0.076,-0.053
Majority baseline,0.700,0.700,1.000,0.412,1.500,0.000,0.000,0.000


In [11]:
glove_size_results = {}
for hidden_size in [8, 50, 200]:
    size_pred = net_oof_predictions(glove_feats, y, cv=5,
                                    hidden_units=hidden_size)
    score = f1_score(y, size_pred, average="macro")
    glove_size_results[hidden_size] = score
    print(f"hidden_units={hidden_size:3d}  macro_f1={score:.3f}")
print("Best hidden size:", max(glove_size_results, key=glove_size_results.get))


hidden_units=  8  macro_f1=0.653
hidden_units= 50  macro_f1=0.649
hidden_units=200  macro_f1=0.647
Best hidden size: 8


**Interpretation (complete after running the cell above):** The hidden-layer size made almost no difference: macro-F1 was 0.653 with 8 hidden units, 0.649 with 50, and 0.647 with 200, so the smallest network was marginally best but the whole spread is only 0.006, which is likely just noise. The GloVe net (macro-F1 0.654, cost 0.931) beats SGD logistic regression (0.623) and k-nearest neighbors (0.570), but is below Linear SVM (0.683) and random forest (0.675). Describing a table as a sentence and averaging word vectors loses information because averaging throws away word order and which value belongs to which field, so the average still contains the words "duration" and "12" but not the fact that they go together, and numbers like 1169 are not meaningful to GloVe at all. With only 1,000 applicants, a wider network also has more parameters to estimate and does not generalize better.

## Task 4: A small neural net on BERT

BERT is a much stronger way to turn text into a vector. In `setup`, BERT was loaded and **frozen** (its 109 million parameters are not trained), so you only use it to read each applicant's description. The vector for the first token, `[CLS]`, is a 768-dimensional summary of the whole text. Then you train the **same small net** from Task 3 on top of it.

**(a)** Write `bert_embeddings(texts, batch_size=32)`. Process the texts in batches. For each batch: tokenize with `tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)`, run `bert(**encoded)` inside `torch.no_grad()`, and keep `out.last_hidden_state[:, 0, :]` (the `[CLS]` vector of every text). Return one numpy array of shape `(len(texts), 768)`. This takes a minute or two.

**(b)** Compute the BERT vectors for all applicants' descriptions, get `net_oof_predictions(bert_feats, y)`, and add a row named `"Neural net on BERT"` to `results`. Print the full table with all the models.

In [12]:
def bert_embeddings(texts, batch_size=32):
    """Return frozen-BERT [CLS] embeddings for applicant descriptions.

    Parameters
    ----------
    texts : sequence of str
        Applicant descriptions in the same order as the target labels.
    batch_size : int, default 32
        Number of descriptions processed in each BERT forward pass.

    Returns
    -------
    numpy.ndarray, shape (len(texts), 768)
        Float32 final-layer hidden-state vector for the [CLS] token.

    Notes
    -----
    Uses the callable tokenizer API from Transformers 4.x. BERT parameters
    remain frozen, and inference runs without gradient tracking.
    """
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")
    if len(texts) == 0:
        return np.empty((0, 768), dtype=np.float32)

    all_vecs = []
    bert.eval()

    for start in range(0, len(texts), batch_size):
        batch = [str(text) for text in texts[start:start + batch_size]]
        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )

        with torch.no_grad():
            output = bert(**encoded)

        cls_vectors = output.last_hidden_state[:, 0, :]
        all_vecs.append(cls_vectors.detach().cpu().numpy())

    return np.vstack(all_vecs).astype(np.float32)


In [13]:
bert_feats = bert_embeddings(texts, batch_size=32)
print("BERT feature matrix:", bert_feats.shape)
pred_bert = net_oof_predictions(bert_feats, y, cv=5)
results.loc["Neural net on BERT"] = pd.Series(
    summarize_predictions(y, pred_bert, group))
results = results.sort_values("macro_f1", ascending=False)
print("All seven models:")
display(results.round(3))
print("GloVe macro-F1:", f"{results.loc['Neural net on GloVe', 'macro_f1']:.3f}")
print("BERT macro-F1:", f"{results.loc['Neural net on BERT', 'macro_f1']:.3f}")
print("Best classic macro-F1:",
      f"{results.drop(index=['Neural net on GloVe', 'Neural net on BERT'], errors='ignore')['macro_f1'].max():.3f}")


BERT feature matrix: (1000, 768)
All seven models:


,accuracy,precision,recall,macro_f1,cost_per_applicant,dp_gap,tpr_gap,fpr_gap
Linear SVM,0.754,0.794,0.876,0.683,0.882,-0.150,-0.170,-0.007
Random forest,0.763,0.782,0.917,0.675,0.953,-0.129,-0.087,-0.090
Neural net on BERT,0.721,0.794,0.811,0.663,0.867,-0.162,-0.096,-0.162
Neural net on GloVe,0.725,0.782,0.841,0.654,0.931,-0.080,-0.092,0.034
Logistic regression (SGD),0.708,0.764,0.844,0.623,1.024,-0.105,-0.108,-0.025
k-nearest neighbors,0.728,0.736,0.953,0.570,1.228,-0.087,-0.076,-0.053
Majority baseline,0.700,0.700,1.000,0.412,1.500,0.000,0.000,0.000


GloVe macro-F1: 0.654
BERT macro-F1: 0.663
Best classic macro-F1: 0.683


**Interpretation (use the printed scores above):** Yes, BERT's macro-F1 (0.663) is slightly higher than GloVe's (0.654), but it does not beat the best classic model, Linear SVM (0.683), or random forest (0.675). BERT does have the lowest cost of all models in the table (0.867 versus 0.882 for Linear SVM), but it also has the largest negative `dp_gap` (-0.162) and `fpr_gap` (-0.162), so in this run it looks like the least fair of the seven models to young applicants, although the under-25 group is small (149 applicants), so these estimates are noisy. BERT gives richer contextual vectors than averaged GloVe, but the input here is a table turned into a templated sentence, and the frozen [CLS] vector is not trained for this task and does not represent numbers like loan amount and duration well. The classic models get the numeric and categorical columns directly, which suits a small tabular dataset better.

## Task 5: Does hiding age help?

A tempting fix: delete `age` so the models cannot see it. The helper below runs **every model** (classic and neural) with some columns left out. For the neural nets, those columns are left out of the text. It takes a few minutes for each call.

Call it on three versions of the features:

1. all features: `run_all()`,
2. without `age`: `run_all(skip=["age"])`, and
3. without `age` plus **three more features** that your Task 1 audit suggested are related to age (for example `employment`, `housing`, `residence_since`, `job`, or `property_magnitude`).

Show **one table** with a row for every model and these columns: `dp_gap_all`, `dp_gap_no_age`, `dp_gap_no_proxies`, `macro_f1_all`, `macro_f1_no_age`, `macro_f1_no_proxies`.

In [14]:
def run_all(skip=()):
    """Evaluate all five classic and both neural models after dropping columns.

    Parameters
    ----------
    skip : iterable of str, default=()
        Feature-column names to remove from the classic feature table and
        from each applicant's text description for the neural models.

    Returns
    -------
    pandas.DataFrame
        Seven-model out-of-fold metric table sorted by macro-F1. Fairness
        groups are retained separately so removing a feature does not erase
        the ability to audit predictions by age group.

    Notes
    -----
    All feature preprocessing and neural-network scaling are refitted within
    each training fold. The same deterministic stratified folds are used for
    each feature version, allowing a fair comparison across versions.
    """
    skip = list(skip)
    X_reduced = X.drop(columns=skip, errors="raise")
    table = compare_models(build_models(), X_reduced, y, group, cv=5)
    reduced_texts = [applicant_to_text(row, skip=skip)
                     for _, row in X.iterrows()]
    reduced_glove = np.vstack([average_glove(t, GLOVE) for t in reduced_texts])
    pred_g = net_oof_predictions(reduced_glove, y, cv=5)
    table.loc["Neural net on GloVe"] = pd.Series(
        summarize_predictions(y, pred_g, group))
    reduced_bert = bert_embeddings(reduced_texts)
    pred_b = net_oof_predictions(reduced_bert, y, cv=5)
    table.loc["Neural net on BERT"] = pd.Series(
        summarize_predictions(y, pred_b, group))
    return table.sort_values("macro_f1", ascending=False)


In [15]:
PROXIES = ["age", "employment", "housing", "residence_since"]

# Three feature sets: all columns, age removed, and age plus three plausible
# age-correlated proxy features removed.
table_all = run_all()
table_no_age = run_all(skip=["age"])
table_no_proxy = run_all(skip=PROXIES)

# Align by model name so every row compares the same model across conditions.
model_order = list(table_all.index)
summary = pd.DataFrame({
    "dp_gap_all": table_all["dp_gap"],
    "dp_gap_no_age": table_no_age["dp_gap"],
    "dp_gap_no_proxies": table_no_proxy["dp_gap"],
    "macro_f1_all": table_all["macro_f1"],
    "macro_f1_no_age": table_no_age["macro_f1"],
    "macro_f1_no_proxies": table_no_proxy["macro_f1"],
}).reindex(model_order)
print("Age/proxy removal comparison:")
display(summary.round(3))


Age/proxy removal comparison:


,dp_gap_all,dp_gap_no_age,dp_gap_no_proxies,macro_f1_all,macro_f1_no_age,macro_f1_no_proxies
Linear SVM,-0.150,-0.135,-0.058,0.683,0.692,0.678
Random forest,-0.129,-0.092,-0.075,0.675,0.683,0.675
Neural net on BERT,-0.162,-0.142,-0.061,0.663,0.668,0.642
Neural net on GloVe,-0.080,-0.092,-0.072,0.654,0.664,0.652
Logistic regression (SGD),-0.105,-0.099,-0.071,0.623,0.645,0.646
k-nearest neighbors,-0.087,-0.084,-0.040,0.570,0.579,0.607
Majority baseline,0.000,0.000,0.000,0.412,0.412,0.412


**Interpretation (use the comparison table above):** Removing `age` alone changed the gaps only modestly: for example Linear SVM went from -0.150 to -0.135, random forest from -0.129 to -0.092, BERT from -0.162 to -0.142, and the GloVe net actually got slightly worse (-0.080 to -0.092). Also removing `employment`, `housing`, and `residence_since` reduced the gap for every model (Linear SVM -0.058, random forest -0.075, BERT -0.061, GloVe -0.072, SGD -0.071, k-nearest neighbors -0.040), but none reached zero. Macro-F1 barely suffered: removing `age` actually raised it slightly for every model (e.g. Linear SVM 0.683 to 0.692), and removing the proxies kept it about the same or higher than with all features for the classic models (Linear SVM 0.678, random forest 0.675, SGD 0.646, k-nearest neighbors 0.607), with the largest drop for the BERT net (0.663 to 0.642). Removing the age column does not remove the effect of age because other features such as employment length and housing are correlated with age and let the model recover much of the same information, so the model can still treat young applicants differently; this also shows a trade-off, since those features carry real information about credit risk.

In [16]:
# Select a transparent no-proxy candidate using the assignment's asymmetric cost.
# If costs tie, prefer the candidate with the higher macro-F1.
no_proxy_cost = table_no_proxy["cost_per_applicant"]
lowest_cost = no_proxy_cost.min()
candidates = table_no_proxy[no_proxy_cost == lowest_cost]
recommended_model = candidates["macro_f1"].idxmax()
recommended_metrics = table_no_proxy.loc[recommended_model]
print("Suggested no-proxy candidate (lowest cost; macro-F1 breaks ties):", recommended_model)
print(f"Cost/applicant: {recommended_metrics['cost_per_applicant']:.3f}")
print(f"Macro-F1: {recommended_metrics['macro_f1']:.3f}")
print(f"DP gap: {recommended_metrics['dp_gap']:.3f}")
print(f"TPR gap: {recommended_metrics['tpr_gap']:.3f}")
print(f"FPR gap: {recommended_metrics['fpr_gap']:.3f}")


Suggested no-proxy candidate (lowest cost; macro-F1 breaks ties): Linear SVM
Cost/applicant: 0.898
Macro-F1: 0.678
DP gap: -0.058
TPR gap: -0.103
FPR gap: 0.125


## Task 6: Memo to the bank
**Write 3 to 4 sentences:** Which model, trained on which version of the features, would you recommend, and why? Use numbers from your tables. Which gap (`dp_gap`, `tpr_gap`, or `fpr_gap`) do you think matters most for a lending decision? Name one limitation of this analysis (for example, how many young applicants there are, or how old the data is).

*My memo*
 I recommend the Linear SVM trained without `age`, `employment`, `housing`, and `residence_since`: it has the lowest cost among the no-proxy models at 0.898 per applicant (versus 1.500 for the majority baseline and 0.882 with all features), a macro-F1 of 0.678 (versus 0.683) and a demographic parity gap that shrinks from -0.150 to -0.058, so we give up very little accuracy for a fairer decision. The remaining gaps are `tpr_gap` = -0.103 and `fpr_gap` = +0.125, and the `fpr_gap` is a sign flip from the all-features model (-0.007), meaning truly bad young applicants are now approved more often than older ones while truly good young applicants are still approved less often. I think `fpr_gap` matters most for lending because approving a bad loan costs 5 times as much as rejecting a good one, so the bank should review this flip before deploying. A limitation is that only 149 applicants are under 25 (versus 851 older), so the gap estimates are noisy, and the data is from 1994 and may not represent today's applicants.
